In [46]:
import pandas as pd
import sys
sys.path.append('../src')
import transform as tr

In [47]:
df = pd.read_csv('../data/raw/funcionarios.csv', sep=',')
df_departamentos = pd.read_csv('../data/raw/departamentos.csv', sep=',')
df_filiais = pd.read_csv('../data/raw/filiais.csv', sep=',')

In [48]:
df = tr.transform_salary(df)
df = tr.transform_date(df)
df = tr.transform_missing_values(df)

In [49]:
df_completo = pd.merge(
    df,
    df_departamentos, 
    on='id_departamento',
    how='left'
)


df_completo = pd.merge(
    df_completo,
    df_filiais,
    on='id_filial',
    how='left'
)

In [50]:
df_completo

,id_colaborador,nome,genero,nivel,data_admissao,data_promocao,salario_base,percentual_bonus,id_departamento,id_filial,...,processos_actifs,horas_extras_mes,score_satisfacao,home_office,status_atual,nome_departamento,id_chefe_departamento,cidade,estado,id_socio_diretor
0,1,Isabela Carvalho 1,Não-binário,Júnior,2024-09-07,NaT,3834.82,0,10,1,...,14,8,NaN,False,Ativo,Civil,100,Salvador,BA,501
1,2,Samuel Teixeira 2,Não-binário,Pleno,2022-09-05,NaT,7267.80,15,10,2,...,16,8,4.9,True,Desligado,Civil,100,São Paulo,SP,502
2,3,Diego Santos 3,Masculino,Júnior,2025-02-01,NaT,4771.24,1,10,1,...,29,11,4.7,True,Ativo,Civil,100,Salvador,BA,501
3,4,Carla Oliveira 4,Não-binário,Pleno,2020-03-04,NaT,7819.21,14,30,2,...,32,13,3.5,False,Desligado,Corporativo,300,São Paulo,SP,502
4,5,Carla Oliveira 5,Não-binário,Júnior,2025-09-15,NaT,4452.47,5,20,3,...,46,42,2.2,False,Desligado,Trabalhista,200,Rio de Janeiro,RJ,503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645,646,Bruno Souza 646,Masculino,Júnior,2024-05-08,NaT,3519.69,9,10,1,...,28,18,4.7,True,Ativo,Civil,100,Salvador,BA,501
646,647,Diego Santos 647,Não-binário,Pleno,2020-09-23,NaT,6590.93,16,40,2,...,14,10,NaN,True,Desligado,Tributário,400,São Paulo,SP,502
647,648,Olivia Barbosa 648,Feminino,Sócio,2013-03-14,2017-04-03,18228.69,51,30,1,...,15,19,3.4,True,Ativo,Corporativo,300,Salvador,BA,501
648,649,Mariana Rocha 649,Feminino,Pleno,2022-10-25,NaT,6501.22,16,40,2,...,21,8,3.9,True,Desligado,Tributário,400,São Paulo,SP,502


### Investigando desligamentos

In [51]:
burnout_macro = df_completo.groupby('status_atual')[['horas_extras_mes', 'processos_actifs']].agg(['mean', 'median'])
display(burnout_macro)

horas_extras_mes        processos_actifs       
                         mean median             mean median
status_atual                                                
Ativo               18.879493   15.0        30.454545   27.0
Desligado           26.418079   20.0        36.485876   31.0

In [52]:
burnout_detalhado = df_completo.groupby(['nivel', 'status_atual'])[['horas_extras_mes', 'processos_actifs']].mean().unstack()
display(burnout_detalhado)

horas_extras_mes            processos_actifs           
status_atual            Ativo  Desligado            Ativo  Desligado
nivel                                                               
Júnior              17.283262  46.976744        29.922747  56.558140
Pleno               20.484615  20.732759        30.784615  30.991379
Sênior              21.298701  13.944444        32.441558  23.944444
Sócio               18.181818        NaN        28.272727        NaN

In [53]:
analise_gestores = df_completo.groupby('id_reporta_a').agg(
    total_liderados=('status_atual', 'count'),
    taxa_evasao=('status_atual', lambda x: (x == 'Desligado').mean() * 100),
    satisfacao_media=('score_satisfacao', 'mean'),
    horas_extras_medias=('horas_extras_mes', 'mean'),
    processos_medios=('processos_actifs', 'mean')
).reset_index()

analise_gestores = analise_gestores[analise_gestores['id_reporta_a'] != -1]
analise_gestores = analise_gestores.round({
    'taxa_evasao': 1,
    'satisfacao_media': 1,
    'horas_extras_medias': 0,
    'processos_medios': 0
}
)

print(analise_gestores.sort_values(by='taxa_evasao', ascending=False))

   id_reporta_a  total_liderados  taxa_evasao  satisfacao_media  \
2           200               89         31.5               3.5   
4           400              126         31.0               3.5   
5           501               99         30.3               3.4   
3           300               98         28.6               3.4   
1           100              103         28.2               3.4   
6           502              102         22.5               3.5   

   horas_extras_medias  processos_medios  
2                 21.0              33.0  
4                 20.0              31.0  
5                 21.0              33.0  
3                 23.0              33.0  
1                 22.0              34.0  
6                 20.0              31.0  


In [54]:
# Salvar o dataset oficial limpo
df_completo.to_csv('../data/processed/funcionarios_abt.csv', index=False, encoding='utf-8')